In [1]:
"""
Phase 4: Churn Prediction Models

Goal: Build and compare predictive models against the SQL scorecard
baseline (29x separation between Critical and Low tiers).
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)

import xgboost as xgb
import shap

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore', category=FutureWarning)

# Seaborn styling
sns.set_style("whitegrid")
sns.set_palette("husl")

print("Imports complete.")
print(f"pandas {pd.__version__}, scikit-learn (check), xgboost {xgb.__version__}")

Imports complete.
pandas 3.0.3, scikit-learn (check), xgboost 3.2.0


In [2]:
# Load DB credentials from .env (same as ETL script)
load_dotenv('../.env')  # .env is in the project root, one level up

DB_USER = os.getenv("POSTGRES_USER")
DB_PASS = quote_plus(os.getenv("POSTGRES_PASSWORD"))
DB_HOST = os.getenv("POSTGRES_HOST")
DB_PORT = os.getenv("POSTGRES_PORT")
DB_NAME = os.getenv("POSTGRES_DB")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# Test the connection
with engine.connect() as conn:
    result = pd.read_sql("SELECT COUNT(*) AS n FROM customers", conn)
    print(f"✓ Connected to PostgreSQL — {result['n'][0]:,} customers in database")

✓ Connected to PostgreSQL — 7,043 customers in database


In [3]:
# Pull the full modeling dataset by joining the 5 normalized tables
modeling_query = """
SELECT
    c.customer_id,

    -- Demographics
    c.gender,
    c.senior_citizen,
    c.has_partner,
    c.has_dependents,

    -- Subscription
    sub.tenure_months,
    sub.contract_type,
    sub.paperless_billing,
    sub.payment_method,

    -- Services
    s.phone_service,
    s.multiple_lines,
    s.internet_service,
    s.online_security,
    s.online_backup,
    s.device_protection,
    s.tech_support,
    s.streaming_tv,
    s.streaming_movies,

    -- Billing
    b.monthly_charges,
    b.total_charges,

    -- Target
    cs.churned
FROM       customers     c
JOIN       subscriptions sub ON c.customer_id = sub.customer_id
JOIN       services      s   ON c.customer_id = s.customer_id
JOIN       billing       b   ON c.customer_id = b.customer_id
JOIN       churn_status  cs  ON c.customer_id = cs.customer_id;
"""

df = pd.read_sql(modeling_query, engine)

print(f"Dataset shape: {df.shape}")
print(f"Churn rate: {df['churned'].mean():.1%}")
print(f"\nColumn data types:")
print(df.dtypes)

Dataset shape: (7043, 21)
Churn rate: 26.5%

Column data types:
customer_id              str
gender                   str
senior_citizen          bool
has_partner             bool
has_dependents          bool
tenure_months          int64
contract_type            str
paperless_billing       bool
payment_method           str
phone_service           bool
multiple_lines           str
internet_service         str
online_security          str
online_backup            str
device_protection        str
tech_support             str
streaming_tv             str
streaming_movies         str
monthly_charges      float64
total_charges        float64
churned                 bool
dtype: object


In [4]:
# Verify the data matches our SQL findings
print("Churn rate by contract type (sanity check vs SQL):")
print(df.groupby('contract_type')['churned'].agg(['count', 'sum', 'mean']))

Churn rate by contract type (sanity check vs SQL):
                count   sum      mean
contract_type                        
Month-to-month   3875  1655  0.427097
One year         1473   166  0.112695
Two year         1695    48  0.028319


In [5]:
# Separate target from features
y = df['churned'].astype(int)  # convert bool to 0/1 for sklearn

# Drop the target and the ID column from features
X = df.drop(columns=['churned', 'customer_id'])

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"\nClass balance:")
print(y.value_counts(normalize=True).rename({0: 'Retained', 1: 'Churned'}).apply(lambda x: f"{x:.1%}"))

Feature matrix X shape: (7043, 19)
Target vector y shape: (7043,)

Class balance:
churned
Retained    73.5%
Churned     26.5%
Name: proportion, dtype: str


In [6]:
# Identify categorical (string) columns that need encoding
categorical_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
print(f"Categorical columns to encode ({len(categorical_cols)}):")
for col in categorical_cols:
    n_unique = X[col].nunique()
    print(f"  {col:20} ({n_unique} unique values)")

# One-hot encode them, dropping the first category of each to avoid
# multicollinearity (the "dummy variable trap")
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print(f"\nBefore encoding: {X.shape[1]} features")
print(f"After encoding:  {X_encoded.shape[1]} features")
print(f"\nFirst few new columns: {X_encoded.columns.tolist()[:5]}")

Categorical columns to encode (11):
  gender               (2 unique values)
  contract_type        (3 unique values)
  payment_method       (4 unique values)
  multiple_lines       (3 unique values)
  internet_service     (3 unique values)
  online_security      (3 unique values)
  online_backup        (3 unique values)
  device_protection    (3 unique values)
  tech_support         (3 unique values)
  streaming_tv         (3 unique values)
  streaming_movies     (3 unique values)

Before encoding: 19 features
After encoding:  30 features

First few new columns: ['senior_citizen', 'has_partner', 'has_dependents', 'tenure_months', 'paperless_billing']


In [7]:
# Split into training and test sets
# - 80% train, 20% test
# - stratify=y preserves the 26.5% churn rate in both splits
# - random_state=42 makes the split reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Training set:  {X_train.shape[0]:,} customers ({X_train.shape[1]} features)")
print(f"Test set:      {X_test.shape[0]:,} customers")
print(f"\nTrain churn rate: {y_train.mean():.1%}")
print(f"Test churn rate:  {y_test.mean():.1%}")
print(f"(should both be ~26.5%, confirming stratification worked)")

Training set:  5,634 customers (30 features)
Test set:      1,409 customers

Train churn rate: 26.5%
Test churn rate:  26.5%
(should both be ~26.5%, confirming stratification worked)


In [8]:
# Logistic Regression is sensitive to feature scale — a feature ranging
# 0-100 will dominate one ranging 0-1. We scale numeric features to have
# mean=0, std=1.
#
# Tree-based models (Random Forest, XGBoost) are scale-invariant and
# don't need this step, but it doesn't hurt them either.

numeric_cols = ['tenure_months', 'monthly_charges', 'total_charges']

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# IMPORTANT: fit the scaler on training data only, then apply to test.
# Fitting on test data would leak test set info into training (a subtle bug).
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols]  = scaler.transform(X_test[numeric_cols])

print("Scaled feature summary (training set):")
print(X_train_scaled[numeric_cols].describe().round(3))

Scaled feature summary (training set):
       tenure_months  monthly_charges  total_charges
count       5634.000         5634.000       5634.000
mean           0.000            0.000         -0.000
std            1.000            1.000          1.000
min           -1.314           -1.537         -1.004
25%           -0.948           -0.967         -0.832
50%           -0.134            0.191         -0.391
75%            0.924            0.842          0.666
max            1.615            1.803          2.830
